In [1]:
from pathlib import Path
import pandas as pd


DATA_PATH = Path("../data/raw/digikala-comments.csv")
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("File size (MB):", round(DATA_PATH.stat().st_size / (1024 ** 2), 2))

Dataset exists: True
File size (MB): 1219.3


In [2]:


sample = pd.read_csv(
    DATA_PATH,
    nrows=10,
    encoding="utf-8"
)

print("Columns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nSample:")
display(sample.head(10))

Columns:
['id', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate']

Data types:
id                         int64
title                        str
body                         str
created_at                   str
rate                     float64
recommendation_status        str
is_buyer                    bool
product_id                 int64
advantages                   str
disadvantages                str
likes                      int64
dislikes                   int64
seller_title                 str
seller_code                  str
true_to_size_rate        float64
dtype: object

Sample:


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,0,0,دیجی‌کالا,5A52N,NaN
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],1,0,دیجی‌کالا,5A52N,NaN
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,NaN,NaN,0,0,آرالیا بیوتی,ADM47,NaN
3,18628562,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,28 اسفند 1399,0.0,recommended,True,3331329,NaN,NaN,0,0,اینجاست آ,9ZMCZ,NaN
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.0,recommended,True,3255700,NaN,NaN,0,0,گالری آرایشی به سیما,CDWHA,NaN
5,53266157,خوبه,قبلا هم استفاده کردم اگه بلد باشین کار کردن با...,11 شهریور 1402,5.0,recommended,True,3305270,NaN,NaN,0,0,یانگوم,ERKSP,NaN
6,46206235,بدک نیست,خوبه,8 اسفند 1401,2.0,NaN,True,3480048,NaN,NaN,0,0,بازرگانی سوشیان,CKWY4,NaN
7,40597171,پد شستشوی براش,خیلی به کارتون میاد قیمتشم مناسبه,24 مهر 1401,5.0,recommended,True,3480048,"['جنسش خوبه\r', 'خوش رنگه\r', 'کاربردیه\r', 'ق...",['نداره '],1,0,بازرگانی سوشیان,CKWY4,NaN
8,46198656,لنز ناخون,دقیقا مطابق عکس بود,7 اسفند 1401,5.0,recommended,True,7626372,NaN,NaN,0,0,هستی نیل,E2C2D,NaN
9,46901811,خوب,برا حالت دادن با اسپری یا پودر عالیه,20 اسفند 1401,5.0,recommended,True,821812,NaN,NaN,0,0,پنی شاپ,CN6G3,NaN


In [3]:
X = sample["body"].tolist()
X

['به درد نمیخوره ',
 'می\u200cتونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه',
 'بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم',
 'به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد من برای صابون ابرو استفاده میکنم راضی ام',
 'معمولیه اگه واسه خونه رنگ کردن شخصی میخواین اوکیه ولی اگه واسه کار دکلره میخواین نه زیاد ولی خوبه',
 'قبلا هم استفاده کردم اگه بلد باشین کار کردن باهاش رو \r\nبرای ی پاکسازی پوست سطحی توی خونه عالیه',
 'خوبه',
 'خیلی به کارتون میاد قیمتشم مناسبه',
 'دقیقا مطابق عکس بود',
 'برا حالت دادن با اسپری یا پودر عالیه']

In [4]:
# شمارش ردیف‌ها و بررسی کیفیت داده ‌ها

from collections import Counter

DATA_PATH = "../data/raw/digikala-comments.csv"

CHUNK_SIZE = 100_000

total_rows = 0

missing_counts = Counter()
recommendation_counts = Counter()
rating_counts = Counter()

empty_body_count = 0
duplicate_id_count = 0
seen_ids = set()

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding="utf-8",
):
    total_rows += len(chunk)

    # Count missing values
    missing_counts.update(
        chunk.isna().sum().to_dict()
    )

    # Count recommendation labels
    recommendation_counts.update(
        chunk["recommendation_status"]
        .fillna("<MISSING>")
        .value_counts()
        .to_dict()
    )

    # Count rating values
    rating_counts.update(
        chunk["rate"]
        .fillna("<MISSING>")
        .value_counts()
        .to_dict()
    )

    # Empty review texts
    empty_body_count += (
        chunk["body"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

    # Duplicate IDs across the dataset
    ids = chunk["id"]

    duplicate_id_count += int(
        ids.isin(seen_ids).sum()
    )

    seen_ids.update(ids.dropna().tolist())

print("Total rows:", total_rows)

print("\nMissing values:")
print(
    pd.Series(missing_counts)
    .sort_values(ascending=False)
)

print("\nRecommendation distribution:")
print(
    pd.Series(recommendation_counts)
    .sort_values(ascending=False)
)

print("\nRating distribution:")
print(
    pd.Series(rating_counts)
    .sort_index()
)

print("\nEmpty review texts:", empty_body_count)

print("Duplicate IDs:", duplicate_id_count)

Total rows: 6156289

Missing values:
true_to_size_rate        6065516
disadvantages            5744017
advantages               5448084
title                    2865062
recommendation_status     894426
seller_title              302181
seller_code               302181
body                         637
rate                           0
created_at                     0
id                             0
is_buyer                       0
product_id                     0
likes                          0
dislikes                       0
dtype: int64

Recommendation distribution:
recommended        4162839
<MISSING>           894426
no_idea             594121
not_recommended     504903
dtype: int64

Rating distribution:
0.00        529098
0.05            97
0.10            66
0.15           181
0.20           661
            ...   
4.85          1476
4.88             2
4.90           426
5.00       2398801
2500.00          1
Length: 178, dtype: int64

Empty review texts: 637
Duplicate IDs: 2682


In [6]:
# Rate_band

DATA_PATH = "../data/raw/digikala-comments.csv"

CHUNK_SIZE = 100_000

rating_band_counts = Counter()
invalid_rate_counts = Counter()

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding="utf-8",
):
    rate = chunk["rate"]

    # Identify invalid ratings
    invalid_mask = (rate < 0) | (rate > 5)

    invalid_rate_counts.update(
        rate[invalid_mask].value_counts().to_dict()
    )

    # Create rating bands
    low_mask = (rate >= 0) & (rate < 2.5)
    medium_mask = (rate >= 2.5) & (rate < 4)
    high_mask = (rate >= 4) & (rate <= 5)

    rating_band_counts.update({
        "low": int(low_mask.sum()),
        "medium": int(medium_mask.sum()),
        "high": int(high_mask.sum()),
    })

print("Rating band distribution:")
print(pd.Series(rating_band_counts))

print("\nInvalid rating values:")
print(pd.Series(invalid_rate_counts))

Rating band distribution:
low       1092329
medium    1059054
high      4004905
dtype: int64

Invalid rating values:
2500.0    1
dtype: int64


In [7]:

DATA_PATH = "../data/raw/digikala-comments.csv"
CHUNK_SIZE = 100_000

counts = {
    "both_valid": 0,
    "recommendation_only": 0,
    "rating_only": 0,
    "neither_valid": 0,
    "empty_body": 0,
}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding="utf-8",
):
    # Is the review text usable?
    valid_body = (
        chunk["body"].notna()
        & chunk["body"].str.strip().ne("")
    )

    # Is the recommendation label available?
    valid_recommendation = (
        chunk["recommendation_status"].notna()
        & chunk["recommendation_status"].isin(
            ["recommended", "not_recommended", "no_idea"]
        )
    )

    # Is the rating within the expected range?
    valid_rating = chunk["rate"].between(0, 5)

    both = valid_recommendation & valid_rating
    recommendation_only = valid_recommendation & ~valid_rating
    rating_only = ~valid_recommendation & valid_rating
    neither = ~valid_recommendation & ~valid_rating

    counts["both_valid"] += int((both & valid_body).sum())

    counts["recommendation_only"] += int(
        (recommendation_only & valid_body).sum()
    )

    counts["rating_only"] += int(
        (rating_only & valid_body).sum()
    )

    counts["neither_valid"] += int(
        (neither & valid_body).sum()
    )

    counts["empty_body"] += int((~valid_body).sum())

print(pd.Series(counts).to_string())

print("\nTotal classified rows:", sum(counts.values()))


both_valid             5261361
recommendation_only          1
rating_only             894290
neither_valid                0
empty_body                 637

Total classified rows: 6156289


both_valid هر دو برچسب معتبر هستند.

recommendation_status + rating_band

In [9]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

In [10]:
# Test split data 
from src.splits import assign_splits

test_data = pd.DataFrame({
    "product_id": [
        100, 100, 100,
        200, 200,
        300, 400, 500,
    ]
})

test_data["split"] = assign_splits(
    test_data["product_id"]
)

display(test_data)

# Every product must belong to exactly one split.
groups_per_product = (
    test_data.groupby("product_id")["split"]
    .nunique()
)

assert groups_per_product.max() == 1

# Repeated execution must produce identical results.
repeated = assign_splits(test_data["product_id"])

assert test_data["split"].equals(repeated)

print("All split tests passed.")

,product_id,split
0,100,test
1,100,test
2,100,test
3,200,train
4,200,train
5,300,train
6,400,train
7,500,train


All split tests passed.
